<a href="https://colab.research.google.com/github/Sanim27/Torch/blob/main/UNET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import torch
import torch.nn as nn

In [39]:
def double_conv(in_c,out_c):
  conv=nn.Sequential(
      nn.Conv2d(in_c,out_c,kernel_size=3),
      nn.ReLU(inplace=True),
      nn.Conv2d(out_c,out_c,kernel_size=3),
      nn.ReLU(inplace=True)
  )
  return conv

In [40]:
def crop_image(tensor,target_tensor):
  target_size=target_tensor.size()[2]
  tensor_size=tensor.size()[2]
  delta=tensor_size-target_size
  delta=delta//2
  return tensor[:,:,delta:tensor_size-delta,delta:tensor_size-delta]

In [41]:
class UNet(nn.Module):
  def __init__(self):
    super(UNet,self).__init__()
    self.max_pool_2x2=nn.MaxPool2d(kernel_size=2,stride=2)
    self.down_conv1=double_conv(1,64)
    self.down_conv2=double_conv(64,128)
    self.down_conv3=double_conv(128,256)
    self.down_conv4=double_conv(256,512)
    self.down_conv5=double_conv(512,1024)

    self.up_trans1=nn.ConvTranspose2d(in_channels=1024,out_channels=512,kernel_size=2,stride=2)
    self.up_conv1=double_conv(1024,512)

    self.up_trans2=nn.ConvTranspose2d(in_channels=512,out_channels=256,kernel_size=2,stride=2)
    self.up_conv2=double_conv(512,256)

    self.up_trans3=nn.ConvTranspose2d(in_channels=256,out_channels=128,kernel_size=2,stride=2)
    self.up_conv3=double_conv(256,128)

    self.up_trans4=nn.ConvTranspose2d(in_channels=128,out_channels=64,kernel_size=2,stride=2)
    self.up_conv4=double_conv(128,64)


    self.out=nn.Conv2d(in_channels=64,out_channels=2,kernel_size=1)


  def forward(self,image):
    # batch_size,channel,height,width
    x1=self.down_conv1(image)  #
    x2=self.max_pool_2x2(x1)
    x3=self.down_conv2(x2)  #
    x4=self.max_pool_2x2(x3)
    x5=self.down_conv3(x4)    #
    x6=self.max_pool_2x2(x5)
    x7=self.down_conv4(x6)    #
    x8=self.max_pool_2x2(x7)
    x9=self.down_conv5(x8)

    x=self.up_trans1(x9)
    y=crop_image(x7,x)
    x=self.up_conv1(torch.cat([x,y],1))

    x=self.up_trans2(x)
    y=crop_image(x5,x)
    x=self.up_conv2(torch.cat([x,y],1))

    x=self.up_trans3(x)
    y=crop_image(x3,x)
    x=self.up_conv3(torch.cat([x,y],1))

    x=self.up_trans4(x)
    y=crop_image(x1,x)
    x=self.up_conv4(torch.cat([x,y],1))

    x=self.out(x)

    print(x.size())
    return x

In [42]:
if __name__=="__main__":
  image=torch.rand((1,1,572,572))
  model=UNet()
  print(model(image))

torch.Size([1, 2, 388, 388])
tensor([[[[-0.0911, -0.0991, -0.0962,  ..., -0.0986, -0.0972, -0.0962],
          [-0.0977, -0.0962, -0.0983,  ..., -0.0929, -0.0964, -0.0928],
          [-0.0964, -0.0997, -0.1001,  ..., -0.0934, -0.0977, -0.0988],
          ...,
          [-0.0917, -0.0959, -0.0959,  ..., -0.0957, -0.0957, -0.0964],
          [-0.0969, -0.0927, -0.0932,  ..., -0.0944, -0.0973, -0.0947],
          [-0.0965, -0.0966, -0.0939,  ..., -0.0959, -0.0961, -0.0983]],

         [[-0.1178, -0.1111, -0.1158,  ..., -0.1137, -0.1145, -0.1138],
          [-0.1140, -0.1152, -0.1161,  ..., -0.1148, -0.1145, -0.1152],
          [-0.1121, -0.1138, -0.1141,  ..., -0.1147, -0.1173, -0.1163],
          ...,
          [-0.1215, -0.1203, -0.1199,  ..., -0.1184, -0.1103, -0.1164],
          [-0.1196, -0.1183, -0.1231,  ..., -0.1174, -0.1160, -0.1155],
          [-0.1235, -0.1205, -0.1167,  ..., -0.1149, -0.1126, -0.1163]]]],
       grad_fn=<ConvolutionBackward0>)
